# Structured Output
We can request LLM to provide the response in specific structure so that processing the response in subsequent steps will be easier.
LangChain supports multiple schema types and methods for enforcing structured output

## Pydantic
Provides the rich feature set with field validation, descriptions and nested structures

In [2]:
import os
from dotenv import load_dotenv
load_dotenv()
from langchain.chat_models import init_chat_model
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")
model = init_chat_model("groq:qwen/qwen3-32b")
model

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.4.8', 'langchain': '1.3.11'}}, output_version=None, profile={'name': 'Qwen3 32B', 'release_date': '2024-12-23', 'last_updated': '2024-12-23', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 40960, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x000002C2D43BC690>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000002C2D43BD090>, model_name='qwen/qwen3-32b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

In [ ]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    # if the datatype of the below variables doesn't match with response of the LLM for example if title response from LLM is integer then it fails as Pydantic validates the fields types
    title: str = Field(description="The title of the Movie")
    year: int = Field(description="Movie Release Year")
    director: str = Field(description="Name of the person who directed the Movie")
    rating: float = Field(description="Movie rating out of 10")

In [4]:
model_with_structure = model.with_structured_output(Movie)
model_with_structure

_ChatModelBinding(bound=ChatGroq(metadata={'lc_versions': {'langchain-core': '1.4.8', 'langchain': '1.3.11'}}, output_version=None, profile={'name': 'Qwen3 32B', 'release_date': '2024-12-23', 'last_updated': '2024-12-23', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 40960, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x000002C2D43BC690>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000002C2D43BD090>, model_name='qwen/qwen3-32b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Movie', 'description': '', 'parameters': {'properties': {'title':

In [6]:
# Without Structure: i.e., default response
model.invoke("Provide details of the Movie Bahubali 2 the Conclusion")

AIMessage(content='<think>\nOkay, so I need to provide details about the movie Bahubali 2: The Conclusion. Let me start by recalling what I know about this film. I remember it\'s a big Indian movie, part of a series that started with Bahubali: The Beginning. Both directed by S.S. Rajamouli, right? It\'s a fantasy action film with a lot of visuals and epic battles.\n\nFirst, the main plot. The first movie ends with Bahubali, played by Prabhas, surviving an assassination attempt and being taken to a hidden location. Then he trains to become a warrior. The second movie is the conclusion, so it should wrap up the story. There\'s a rivalry between two kingdoms, maybe Mahishmati and some other place. The main antagonist is probably Kattappa, who I think is the one who killed Bahubali in the first movie but there\'s a twist. Also, there\'s the character of Devasena, played by Tamannaah, who is Bahubali\'s wife. Then there\'s Shruti Haasan as Avanthika, who is the daughter of the king of anoth

In [ ]:
# With Pydantic Structure
model_with_structure.invoke("Provide details of the Movie Bahubali 2 the Conclusion")

Movie(title='Bahubali 2 the Conclusion', year=2017, director='S.S. Rajamouli', rating=8.5)

## Message Output with Parsed Structure

In [7]:
model_with_structure = model.with_structured_output(Movie, include_raw=True)
model_with_structure.invoke("Provide details of the Movie Bahubali 2 the Conclusion")

{'raw': AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user is asking for details about the movie "Bahubali 2: The Conclusion." Let me see what I need to do here. The tools provided include a Movie function with parameters like director, rating, title, and year. All these are required.\n\nFirst, I need to recall the information about Bahubali 2. The title is clear: "Bahubali 2: The Conclusion." The director is S.S. Rajamouli. The release year was 2017. As for the rating, I think it\'s around 8.5 or 9 on IMDb. Let me confirm that. Yes, it\'s 8.5/10.\n\nNow, I need to structure this into the required JSON format. Check the parameters: title, year, director, rating. All are required. The title should include the subtitle "The Conclusion." The director\'s name is S.S. Rajamouli. Year is 2017. Rating is 8.5. \n\nWait, the rating is a number, so in JSON, it should be a number, not a string. Make sure to format that correctly. Also, the title should be the full title

## Nested Structure

In [17]:
from pydantic import BaseModel, Field

class Actor(BaseModel):
    name: str
    role: str

class Movie(BaseModel):
    # if the datatype of the below variables doesn't match with response of the LLM for example if title response from LLM is integer then it fails as Pydantic validates the fields types
    title: str = Field(description="The title of the Movie")
    year: int = Field(description="Movie Release Year")
    director: str = Field(description="Name of the person who directed the Movie")
    rating: float = Field(description="Movie rating out of 10")
    cast: list[Actor]

In [18]:
model_with_structure = model.with_structured_output(Movie)
model_with_structure.invoke("Provide details of the Movie Bahubali 2 the Conclusion")

Movie(title='Bahubali 2: The Conclusion', year=2017, director='S. S. Rajamouli', rating=9.0, cast=[Actor(name='Prabhas', role='Shivudu / Bahubali'), Actor(name='Rana Daggubati', role='Bhallala Deva'), Actor(name='Tamannaah Bhatia', role='Devasena'), Actor(name='Anushka Shetty', role='Avanthika'), Actor(name='Sathyaraj', role='Mahendra Baahubali')])

In [20]:
model.profile

{'name': 'Qwen3 32B',
 'release_date': '2024-12-23',
 'last_updated': '2024-12-23',
 'open_weights': True,
 'max_input_tokens': 131072,
 'max_output_tokens': 40960,
 'text_inputs': True,
 'image_inputs': False,
 'audio_inputs': False,
 'video_inputs': False,
 'text_outputs': True,
 'image_outputs': False,
 'audio_outputs': False,
 'video_outputs': False,
 'reasoning_output': True,
 'tool_calling': True,
 'attachment': False,
 'temperature': True}

## Typed Dict (Alternative of PyDantic)
TypedDict provides a simpler alternative using Python's built-in typing, ideal when you don' need runtime validation

In [15]:
from typing_extensions import TypedDict, Annotated

class MovieDict(TypedDict):
    # if the datatype of the below variables doesn't match with response of the LLM for example if title response from LLM is integer then it fails as Pydantic validates the fields types
    title: Annotated[str, "The title of the Movie"]
    director: Annotated[str, "Name of the person who directed the Movie"]
    box_office_collection: Annotated[float, "Global Box Office collection of the movie in INR in crores"]
    

In [16]:
model_with_structure = model.with_structured_output(MovieDict)
model_with_structure.invoke("Provide details of the Movie Bahubali 2 the Conclusion")

{'box_office_collection': 1300,
 'director': 'S.S. Rajamouli',
 'title': 'Bahubali 2: The Conclusion'}

## Data Classes
They don't have any validations like Pydantic, @dataclass should be used
- A Simple demonstration of Agent with Structured Output

In [24]:
from dataclasses import dataclass
from langchain.agents import create_agent

@dataclass
class ContactInfo:
    name: str # Name of the person
    phone: str # Contact number of the person
    email: str # Email address of the person

agent = create_agent(
    model='google_genai:gemini-2.5-flash-lite',
    response_format = ContactInfo
)

result = agent.invoke({
    "messages": [{"role":"user", "content": "Extract contact information from : Pawan Kalyan, pawankalyan@deputycm.com, 9999999999"}]
})
print(result["structured_response"])

ContactInfo(name='Pawan Kalyan', phone='9999999999', email='pawankalyan@deputycm.com')


In [22]:
from pydantic import BaseModel, Field
from langchain.agents import create_agent

class ContactInfo(BaseModel):
    name: str = Field(description="Name of the person")
    phone: str = Field(description="Contact number of the person")
    email: str = Field(description="Email address of the person")

agent = create_agent(
    model='google_genai:gemini-2.5-flash-lite',
    response_format = ContactInfo
)

result = agent.invoke({
    "messages": [{"role":"user", "content": "Extract contact information from : Pawan Kalyan, pawankalyan@deputycm.com, 9999999999"}]
})
print(result["structured_response"])

name='Pawan Kalyan' phone='9999999999' email='pawankalyan@deputycm.com'
